# Задание 1 (5 балла)

Имплементируйте алгоритм Леска (описание есть в семинаре) и оцените качество его работы на датасете `data/corpus_wsd_50k.txt`

В качестве метрики близости вы должны попробовать два подхода:

1) Jaccard score на множествах слов (определений и контекста)
2) Cosine distance на эмбедингах sentence_transformers

В качестве метрики используйте accuracy (% правильных ответов). Предсказывайте только многозначные слова в датасете

Контекст вы можете определить самостоятельно (окно вокруг целевого слова или все предложение). Также можете поэкспериментировать с предобработкой для обоих методов.

In [ ]:
import re
import numpy as np
from nltk.corpus import wordnet as wn
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Загрузка датасета
corpus = open('corpus_wsd_50k.txt').read().split('\n\n')
corpus_wsd = [[s.split('\t') for s in sent.split('\n')] for sent in corpus if sent.strip()]

# Загрузка модели для эмбеддингов
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def preprocess(text):
    """Приводим текст к нижнему регистру, убираем пунктуацию, разбиваем на слоав"""
    return set(re.findall(r'\b[a-zа-яё]+\b', text.lower()))

def jaccard(set1, set2):
    """Jaccard score на множествах слов."""
    union = set1 | set2
    if not union:
        return 0.0
    return len(set1 & set2) / len(union)

def get_synsets_for_word(word, pos=None):
    """Получить список сисетов для слова """
    return wn.synsets(word, pos=pos) if pos else wn.synsets(word)

def lesk_jaccard(context, word, pos=None):
    """Алгоритм Леска с Jaccard score"""
    synsets = get_synsets_for_word(word, pos)
    if not synsets:
        return None
    
    context_words = preprocess(context)
    best_synset = None
    best_score = -1
    
    for synset in synsets:
        definition = synset.definition()
        # Добавляем примеры для улучшения контекста
        examples = ' '.join(synset.examples())
        full_def = definition + ' ' + examples
        def_words = preprocess(full_def)
        
        score = jaccard(context_words, def_words)
        if score > best_score:
            best_score = score
            best_synset = synset
    
    return best_synset

def lesk_cosine(context, word, pos=None):
    """Алгоритм Леска с косинусным расстоянием на эмбедингах"""
    synsets = get_synsets_for_word(word, pos)
    if not synsets:
        return None
    
    context_emb = model.encode([context])
    best_synset = None
    best_score = -1
    
    for synset in synsets:
        definition = synset.definition()
        examples = ' '.join(synset.examples())
        full_def = definition + ' ' + examples
        def_emb = model.encode([full_def])
        
        score = cosine_similarity(context_emb, def_emb)[0][0]
        if score > best_score:
            best_score = score
            best_synset = synset
    
    return best_synset

# Задание 2 (5 балла)
Попробуйте разные алгоритмы кластеризации на датасете - `https://github.com/nlpub/russe-wsi-kit/blob/initial/data/main/wiki-wiki/train.csv`

Используйте код из семинара как основу. Используйте ARI как метрику качества.

Попробуйте все 4 алгоритма кластеризации, про которые говорилось на семинаре. Для каждого из алгоритмов попробуйте настраивать гиперпараметры (посмотрите их в документации). Прогоните как минимум 5 экспериментов (не обязательно успешных) с разными параметрами на каждый алгоритме кластеризации и оцените: качество кластеризации, скорость работы, интуитивность параметров.

Помимо этого также выберите 1 дополнительный алгоритм кластеризации отсюда - https://scikit-learn.org/stable/modules/clustering.html , опишите своими словами принцип его работы  и проделайте аналогичные эксперименты. 

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import (
    KMeans, DBSCAN, AffinityPropagation,
    AgglomerativeClustering, SpectralClustering
)
from sklearn.metrics import adjusted_rand_score
from sentence_transformers import SentenceTransformer


# Загрузка данных
df = pd.read_csv(
    'https://raw.githubusercontent.com/nlpub/russe-wsi-kit/initial/data/main/wiki-wiki/train.csv',
    sep='\t'
)

# Группируем по слову у каждого слова свои контексты и значения
grouped_df = df.groupby('word')[['word', 'context', 'gold_sense_id']]

print(f"Всего контекстов: {len(df)}")
print(f"Уникальных слов: {df['word'].nunique()}")


# Загрузка модели для эмбеддингов
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')


def embed(texts):
    """Создает эмбеддинги для списка текстов и нормализует их."""
    if isinstance(texts, str):
        texts = [texts]
    X = model.encode(texts, show_progress_bar=False, batch_size=16)
    # Нормализуем для косинусной близости
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    return X


print('размерность:', model.get_sentence_embedding_dimension())


# Вспомогательная функция для оценки

def evaluate_algorithm(grouped_df, algorithm_class, params):
    ari_scores = []
    start_time = time.time()
    total_words = 0

    for key, _ in grouped_df:
        group = grouped_df.get_group(key)
        texts = group['context'].values
        true_labels = group['gold_sense_id'].values

        # Пропускаем слова с одним значением
        if len(np.unique(true_labels)) < 2:
            continue

        X = embed(texts)

        try:
            # Инициализируем алгоритм с учётом метрикии
            if algorithm_class.__name__ == 'DBSCAN':
                cluster = algorithm_class(metric='cosine', **params)
            elif algorithm_class.__name__ == 'SpectralClustering':
                cluster = algorithm_class(affinity='cosine', **params)
            else:
                cluster = algorithm_class(**params)

            labels = cluster.fit_predict(X)

            # Для DBSCAN и SpectralClustering шум помечается как -1
            # Заменяем -1 на отдельную метку кластера
            if -1 in labels:
                labels = labels.copy()
                labels[labels == -1] = labels.max() + 1

            ari = adjusted_rand_score(true_labels, labels)
            ari_scores.append(ari)
            total_words += 1

        except Exception as e:
            print(f"  Ошибка для слова '{key}': {e}")
            continue

    elapsed_time = time.time() - start_time
    mean_ari = np.mean(ari_scores) if ari_scores else 0

    return {
        'mean_ari': mean_ari,
        'time': elapsed_time,
        'n_words': total_words,
        'ari_scores': ari_scores
    }


# KMeans

print("\n" + "="*60)
print("ЭКСПЕРИМЕНТЫ С АЛГОРИТМАМИ КЛАСТЕРИЗАЦИИ")
print("="*60)

print("\n--- KMeans ---")
print("Параметры: n_clusters (количество кластеров)")

kmeans_results = []

for n_clusters in [2, 3, 4, 5, 7]:
    params = {'n_clusters': n_clusters, 'n_init': 10, 'random_state': 42}
    result = evaluate_algorithm(grouped_df, KMeans, params)
    kmeans_results.append({
        'params': f"n_clusters={n_clusters}",
        'mean_ari': result['mean_ari'],
        'time': result['time'],
        'n_words': result['n_words']
    })
    print(f"  n_clusters={n_clusters}: ARI={result['mean_ari']:.3f}, "
          f"time={result['time']:.2f}s, words={result['n_words']}")


# DBSCAN

print("\n--- DBSCAN ---")
print("Параметры: eps (радиус), min_samples (мин. соседей)")

dbscan_results = []

for eps in [0.1, 0.2, 0.3, 0.5]:
    for min_samples in [1, 2, 3]:
        params = {'eps': eps, 'min_samples': min_samples}
        result = evaluate_algorithm(grouped_df, DBSCAN, params)
        dbscan_results.append({
            'params': f"eps={eps}, min_samples={min_samples}",
            'mean_ari': result['mean_ari'],
            'time': result['time'],
            'n_words': result['n_words']
        })
        print(f"  eps={eps}, min_samples={min_samples}: ARI={result['mean_ari']:.3f}, "
              f"time={result['time']:.2f}s, words={result['n_words']}")


# Affinity Propagation

print("\n--- Affinity Propagation ---")
print("Параметры: damping (затухание), preference (предпочтение)")

ap_results = []

for damping in [0.5, 0.7, 0.9]:
    for preference in [-30, -50, -100]:
        params = {'damping': damping, 'preference': preference, 'random_state': 42}
        result = evaluate_algorithm(grouped_df, AffinityPropagation, params)
        ap_results.append({
            'params': f"damping={damping}, preference={preference}",
            'mean_ari': result['mean_ari'],
            'time': result['time'],
            'n_words': result['n_words']
        })
        print(f"  damping={damping}, preference={preference}: ARI={result['mean_ari']:.3f}, "
              f"time={result['time']:.2f}s, words={result['n_words']}")


# Agglomerative Clustering

print("\n--- Agglomerative Clustering ---")
print("Параметры: n_clusters (количество кластеров), linkage (тип связи)")

agg_results = []

for n_clusters in [2, 3, 4, 5]:
    params = {'n_clusters': n_clusters, 'linkage': 'ward'}
    result = evaluate_algorithm(grouped_df, AgglomerativeClustering, params)
    agg_results.append({
        'params': f"n_clusters={n_clusters}, linkage=ward",
        'mean_ari': result['mean_ari'],
        'time': result['time'],
        'n_words': result['n_words']
    })
    print(f"  n_clusters={n_clusters}, linkage=ward: ARI={result['mean_ari']:.3f}, "
          f"time={result['time']:.2f}s, words={result['n_words']}")


# Дополнительный алгоритм: Spectral Clustering

print("\n" + "="*60)
print("SPECTRAL CLUSTERING")
print("="*60)

spectral_results = []

for n_clusters in [2, 3, 4, 5]:
    for affinity in ['nearest_neighbors', 'rbf']:
        try:
            params = {'n_clusters': n_clusters, 'affinity': affinity, 'random_state': 42}
            result = evaluate_algorithm(grouped_df, SpectralClustering, params)
            spectral_results.append({
                'params': f"n_clusters={n_clusters}, affinity={affinity}",
                'mean_ari': result['mean_ari'],
                'time': result['time'],
                'n_words': result['n_words']
            })
            print(f"  n_clusters={n_clusters}, affinity={affinity}: ARI={result['mean_ari']:.3f}, "
                  f"time={result['time']:.2f}s, words={result['n_words']}")
        except Exception as e:
            print(f"  n_clusters={n_clusters}, affinity={affinity}: Ошибка - {e}")



# Итоговый анализ

print("\n" + "="*60)
print("ИТОГОВЫЙ АНАЛИЗ")
print("="*60)

# Собираем все результаты в один DataFrame
all_results = []
for r in kmeans_results:
    all_results.append({'algorithm': 'KMeans', **r})
for r in dbscan_results:
    all_results.append({'algorithm': 'DBSCAN', **r})
for r in ap_results:
    all_results.append({'algorithm': 'AffinityPropagation', **r})
for r in agg_results:
    all_results.append({'algorithm': 'Agglomerative', **r})
for r in spectral_results:
    all_results.append({'algorithm': 'Spectral', **r})

results_df = pd.DataFrame(all_results)

# Лучшие результаты по ARI для каждого алгоритма
print("\nЛучшие результаты по ARI для каждого алгоритма:")
print("-" * 70)
for algo in results_df['algorithm'].unique():
    best = results_df[results_df['algorithm'] == algo].sort_values(
        'mean_ari', ascending=False
    ).iloc[0]
    print(f"{algo:25} | {best['params']:35} | ARI={best['mean_ari']:.3f} | "
          f"time={best['time']:.2f}s")

# Лучшие результаты по времени
print("\nСамые быстрые конфигурации:")
print("-" * 70)
fastest = results_df.sort_values('time').head(5)
for _, row in fastest.iterrows():
    print(f"{row['algorithm']:25} | {row['params']:35} | time={row['time']:.2f}s | "
          f"ARI={row['mean_ari']:.3f}")

# Сводная таблица по алгоритмам
print("\nСводка по алгоритмам (средний ARI и среднее время):")
print("-" * 70)
summary = results_df.groupby('algorithm').agg({
    'mean_ari': ['mean', 'max', 'std'],
    'time': ['mean', 'min']
}).round(3)
print(summary)